# SENTINEL — Train UCSD Ped2 từ Google Drive bằng GPU

Notebook này giữ repository, dataset, feature cache và artifacts trên Google Drive. Dataset được giải nén tạm vào `/content` để đọc nhanh. Chạy các cell từ trên xuống; ở những phiên sau chỉ cần mount Drive, chạy cell **Pull code mới nhất**, cài dependency và tiếp tục từ feature cache.

## 0. Cấu hình
Đổi `REPO_URL` nếu bạn dùng fork khác. Các đường dẫn còn lại có thể giữ nguyên.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/TranDuon/anomaly-behavior-detector-system.git"
BRANCH = "main"
DRIVE_ROOT = Path("/content/drive/MyDrive/SENTINEL")
REPO_DIR = DRIVE_ROOT / "repo" / "anomaly-behavior-detector-system"
DATASET_NAME = "ucsd_ped2"
DATASET_DIR = DRIVE_ROOT / "datasets" / DATASET_NAME
DATASET_ARCHIVE = DATASET_DIR / "UCSD_Anomaly_Dataset.tar.gz"
FEATURE_DIR = DRIVE_ROOT / "features" / DATASET_NAME
ARTIFACT_DIR = DRIVE_ROOT / "artifacts" / DATASET_NAME
LOCAL_DATA_ROOT = Path("/content/datasets")

EPOCHS = 50
BATCH_SIZE = 64
STRIDE = 10
SEED = 42
print("Repository:", REPO_DIR)
print("Dataset archive:", DATASET_ARCHIVE)
print("Feature cache:", FEATURE_DIR)
print("Artifacts:", ARTIFACT_DIR)

## 1. Mount Google Drive và tạo cấu trúc thư mục

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

for path in [REPO_DIR.parent, DATASET_DIR, FEATURE_DIR, ARTIFACT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Drive workspace đã sẵn sàng:", DRIVE_ROOT)

## 2. Clone repository lần đầu
Cell này chỉ clone nếu thư mục Drive chưa có `.git`.

In [ ]:
import os
import subprocess

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
    print("Đã clone repository vào Drive.")
else:
    print("Repository đã tồn tại, bỏ qua clone.")

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "--oneline", "-1"], check=True)

## 3. Pull code mới nhất
Sau mỗi lần sửa ở local và push GitHub, chạy cell này. Cell dùng `--ff-only` để không tự tạo merge commit trên Drive. Nếu repository trên Drive có file tracked bị sửa, hãy xử lý hoặc bỏ thay đổi đó trước.

In [ ]:
import os
import subprocess

status = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "status", "--porcelain"],
    text=True,
)
if status.strip():
    raise RuntimeError(
        "Repository trên Drive đang có thay đổi chưa commit:\n" + status
    )

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
subprocess.run(
    ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
    check=True,
)
os.chdir(REPO_DIR)
subprocess.run(["git", "log", "--oneline", "-1"], check=True)

## 4. Cài dependency và kiểm tra GPU

In [ ]:
import subprocess
import sys

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

In [ ]:
import torch

USE_GPU = torch.cuda.is_available()
DEVICE = "cuda" if USE_GPU else "cpu"
print("PyTorch:", torch.__version__)
print("CUDA available:", USE_GPU)
if USE_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CẢNH BÁO: Runtime chưa có GPU; kiểm tra Runtime > Change runtime type.")

(REPO_DIR / ".env").write_text(
    f"YOLO_DEVICE={DEVICE}\n"
    "YOLO_MODEL=yolov8n.pt\n"
    "YOLO_CONF_THRESHOLD=0.40\n",
    encoding="utf-8",
)
print((REPO_DIR / ".env").read_text())

## 5. Đặt dataset lên Drive
Tải `UCSD_Anomaly_Dataset.tar.gz` và đặt đúng tại đường dẫn được in dưới đây. Nên tải/upload file nén một lần; không giải nén hàng nghìn TIFF trực tiếp trên Drive.

In [ ]:
print("Hãy đặt archive tại:")
print(DATASET_ARCHIVE)
print("Đã tồn tại:", DATASET_ARCHIVE.exists())
if DATASET_ARCHIVE.exists():
    print(f"Kích thước: {DATASET_ARCHIVE.stat().st_size / 1024**2:.1f} MB")

### Tùy chọn tải trực tiếp
Chỉ bật cell dưới nếu bạn chưa upload archive lên Drive. Nếu máy chủ UCSD không phản hồi, tải bằng trình duyệt rồi upload thủ công vào đúng đường dẫn.

In [ ]:
DOWNLOAD_DATASET = False
DATASET_URL = "http://www.svcl.ucsd.edu/projects/anomaly/UCSD_Anomaly_Dataset.tar.gz"

if DOWNLOAD_DATASET and not DATASET_ARCHIVE.exists():
    subprocess.run(
        ["wget", "-c", DATASET_URL, "-O", str(DATASET_ARCHIVE)],
        check=True,
    )
print("Archive ready:", DATASET_ARCHIVE.exists())

## 6. Stage dataset sang `/content`
Drive chỉ giữ bản nén bền vững; quá trình đọc TIFF chạy từ ổ local của Colab để nhanh hơn.

In [ ]:
import shutil

if not DATASET_ARCHIVE.exists():
    raise FileNotFoundError(f"Chưa có dataset archive: {DATASET_ARCHIVE}")

LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
local_archive = Path("/content/UCSD_Anomaly_Dataset.tar.gz")
if not local_archive.exists() or local_archive.stat().st_size != DATASET_ARCHIVE.stat().st_size:
    shutil.copy2(DATASET_ARCHIVE, local_archive)

train_candidates = list(LOCAL_DATA_ROOT.rglob("UCSDped2/Train"))
if not train_candidates:
    subprocess.run(["tar", "-xzf", str(local_archive), "-C", str(LOCAL_DATA_ROOT)], check=True)
    train_candidates = list(LOCAL_DATA_ROOT.rglob("UCSDped2/Train"))
if not train_candidates:
    raise RuntimeError("Không tìm thấy UCSDped2/Train sau khi giải nén.")

TRAIN_ROOT = train_candidates[0]
train_clips = sorted(path for path in TRAIN_ROOT.iterdir() if path.is_dir())
print("TRAIN_ROOT:", TRAIN_ROOT)
print("Số clip normal:", len(train_clips))
print("Ba clip đầu:", [path.name for path in train_clips[:3]])

## 7. Smoke test một clip
Chạy tối đa 300 frame để xác nhận YOLO phát hiện người và feature cache được tạo trước khi chạy toàn bộ dataset.

In [ ]:
SMOKE_DIR = FEATURE_DIR / "smoke"
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        sys.executable, "scripts/extract_features.py",
        "--input", str(train_clips[0]),
        "--output-dir", str(SMOKE_DIR),
        "--stride", str(STRIDE),
        "--max-frames", "300",
        "--force",
    ],
    cwd=REPO_DIR,
    check=True,
)

In [ ]:
import json

smoke_manifest = json.loads((SMOKE_DIR / "manifest.json").read_text())
print(json.dumps(smoke_manifest, indent=2))
if not any(item.get("sequences", 0) > 0 for item in smoke_manifest["clips"].values()):
    print("CẢNH BÁO: chưa tạo được sequence. Hãy kiểm tra detection count hoặc giảm YOLO_CONF_THRESHOLD.")

## 8. Trích xuất toàn bộ feature normal
Mỗi clip được cache thành một `.npz`. Nếu runtime ngắt, chạy lại cell này; các clip đã có cache sẽ được bỏ qua.

In [ ]:
TRAIN_FEATURE_DIR = FEATURE_DIR / "train"
TRAIN_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        sys.executable, "scripts/extract_features.py",
        "--input", str(TRAIN_ROOT),
        "--output-dir", str(TRAIN_FEATURE_DIR),
        "--stride", str(STRIDE),
    ],
    cwd=REPO_DIR,
    check=True,
)

## 9. Train LSTM từ feature cache
Script chia validation theo clip, fit scaler chỉ trên clip train, train GPU, calibration threshold trên validation normal và lưu đầy đủ artifacts.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        sys.executable, "scripts/train_features.py",
        "--features-dir", str(TRAIN_FEATURE_DIR),
        "--output-dir", str(ARTIFACT_DIR),
        "--dataset-name", "UCSD Ped2",
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--val-ratio", "0.2",
        "--seed", str(SEED),
        "--device", DEVICE,
    ],
    cwd=REPO_DIR,
    check=True,
)
print("Artifacts:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", path.name, path.stat().st_size, "bytes")

## 10. Cài artifacts vào repository trên Drive
Ba file phải đi cùng nhau: model, scaler và threshold. Chúng nằm trong thư mục ignored `models/`, nên `git pull` không xóa chúng.

In [ ]:
MODEL_DIR = REPO_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
for name in ["lstm_autoencoder.pt", "feature_scaler.npz", "threshold.json"]:
    source = ARTIFACT_DIR / name
    if not source.exists():
        raise FileNotFoundError(source)
    shutil.copy2(source, MODEL_DIR / name)
print("Đã cài artifacts vào:", MODEL_DIR)

## Sau khi local push code mới
Ở phiên Colab hiện tại hoặc phiên sau: mount Drive → chạy cell cấu hình → chạy **Pull code mới nhất** → cài dependency nếu runtime mới → tiếp tục smoke/extract/train. Feature cache và artifacts không bị mất khi runtime reset.